In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [ ]:
DATA_DIR = '/content/olist_data'
OUTPUT_DIR = '/content/cleaned_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
customers   = pd.read_csv('olist_customers_dataset.csv')
orders      = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews  = pd.read_csv('olist_order_reviews_dataset.csv')
products    = pd.read_csv('olist_products_dataset.csv')
sellers     = pd.read_csv('olist_sellers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

raw_frames = {
    'customers': customers, 'orders': orders, 'order_items': order_items,
    'order_payments': order_payments, 'order_reviews': order_reviews,
    'products': products, 'sellers': sellers, 'geolocation': geolocation,
    'category_translation': category_translation
}

for name, df in raw_frames.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols, "
          f"{df.duplicated().sum()} exact duplicate rows,"
          f" {df.isnull().sum().sum()} missing values")

customers: 99441 rows, 5 cols, 0 exact duplicate rows, 0 missing values
orders: 99441 rows, 8 cols, 0 exact duplicate rows, 4908 missing values
order_items: 112650 rows, 7 cols, 0 exact duplicate rows, 0 missing values
order_payments: 103886 rows, 5 cols, 0 exact duplicate rows, 0 missing values
order_reviews: 99224 rows, 7 cols, 0 exact duplicate rows, 145903 missing values
products: 32951 rows, 9 cols, 0 exact duplicate rows, 2448 missing values
sellers: 3095 rows, 4 cols, 0 exact duplicate rows, 0 missing values
geolocation: 1000163 rows, 5 cols, 261831 exact duplicate rows, 0 missing values
category_translation: 71 rows, 2 cols, 0 exact duplicate rows, 0 missing values


In [ ]:
def standardize_text_cols(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            if 'state' in col:
                df[col] = df[col].str.upper()
            elif 'city' in col:
                df[col] = df[col].str.title()
    return df

def to_datetime_cols(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def report_missing(df, name):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing):
        print(f"--- Missing values in {name} ---")
        print(missing)
    else:
        print(f"{name}: no missing values")

## 1. Clean `customers`

In [ ]:
customers_clean = customers.drop_duplicates()
customers_clean = standardize_text_cols(customers_clean, ['customer_city', 'customer_state'])
customers_clean['customer_zip_code_prefix'] = (
    customers_clean['customer_zip_code_prefix'].astype(str).str.zfill(5))
report_missing(customers_clean, 'customers')


customers: no missing values


## 2. Clean `sellers`

In [ ]:
sellers_clean = sellers.drop_duplicates()
sellers_clean = standardize_text_cols(sellers_clean, ['seller_city', 'seller_state'])
sellers_clean['seller_zip_code_prefix'] = (
    sellers_clean['seller_zip_code_prefix'].astype(str).str.zfill(5))
report_missing(sellers_clean, 'sellers')

sellers: no missing values


## 3. Clean `geolocation`


In [ ]:
geo_clean = geolocation.drop_duplicates()
geo_clean = standardize_text_cols(geo_clean, ['geolocation_city', 'geolocation_state'])


geo_clean = geo_clean[
    geo_clean['geolocation_lat'].between(-34, 6) &
    geo_clean['geolocation_lng'].between(-74, -32)
]

geo_clean['geolocation_zip_code_prefix'] = (
    geo_clean['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
)

geo_clean = (
    geo_clean.groupby('geolocation_zip_code_prefix')
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_city=('geolocation_city', lambda s: s.mode().iloc[0]),
        geolocation_state=('geolocation_state', lambda s: s.mode().iloc[0]),
    )
    .reset_index()
)
report_missing(geo_clean, 'geolocation')

geolocation: no missing values


## 4. Clean `product_category_name_translation`

In [ ]:
category_translation_clean = category_translation.drop_duplicates()
category_translation_clean.columns = ['product_category_name', 'product_category_name_english']
report_missing(category_translation_clean, 'category_translation')

category_translation: no missing values


## 5. Clean `products`


In [ ]:
products_clean = products.drop_duplicates()

products_clean = products_clean.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')

products_clean = products_clean.merge(
    category_translation_clean, on='product_category_name', how='left'
)
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english'].fillna('unknown')
)

numeric_cols = [
    'product_name_length', 'product_description_length', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
]
for col in numeric_cols:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

report_missing(products_clean, 'products')

products: no missing values


## 6. Clean `orders`



In [ ]:
orders_clean = orders.drop_duplicates()

date_cols = [
    'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date'
]
orders_clean = to_datetime_cols(orders_clean, date_cols)
orders_clean['order_status'] = orders_clean['order_status'].str.strip().str.lower()

report_missing(orders_clean, 'orders')
print(orders_clean['order_status'].value_counts())

--- Missing values in orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 7. Clean `order_items`

In [ ]:
order_items_clean = order_items.drop_duplicates()
order_items_clean = to_datetime_cols(order_items_clean, ['shipping_limit_date'])
order_items_clean = order_items_clean[
    (order_items_clean['price'] > 0) & (order_items_clean['freight_value'] >= 0)
]
report_missing(order_items_clean, 'order_items')

order_items: no missing values


## 8. Clean `order_payments`

In [ ]:
order_payments_clean = order_payments.drop_duplicates()
order_payments_clean['payment_type'] = order_payments_clean['payment_type'].str.strip().str.lower()
order_payments_clean = order_payments_clean[order_payments_clean['payment_value'] >= 0]
report_missing(order_payments_clean, 'order_payments')
print(order_payments_clean['payment_type'].value_counts())

order_payments: no missing values
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


## 9. Clean `order_reviews`

In [ ]:
order_reviews_clean = order_reviews.drop_duplicates(subset='review_id')
order_reviews_clean = to_datetime_cols(
    order_reviews_clean, ['review_creation_date', 'review_answer_timestamp']
)

# Comment text is optional — keep as empty string rather than dropping the row
order_reviews_clean['review_comment_title'] = order_reviews_clean['review_comment_title'].fillna('')
order_reviews_clean['review_comment_message'] = order_reviews_clean['review_comment_message'].fillna('')
order_reviews_clean = order_reviews_clean[order_reviews_clean['review_score'].between(1, 5)]
report_missing(order_reviews_clean, 'order_reviews')

order_reviews: no missing values


In [ ]:
cleaned_frames = {
    'customers': customers_clean,
    'sellers': sellers_clean,
    'geolocation': geo_clean,
    'category_translation': category_translation_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
    'order_payments': order_payments_clean,
    'order_reviews': order_reviews_clean,
}

for name, df in cleaned_frames.items():
    out_path = f'{OUTPUT_DIR}/{name}_clean.csv'
    df.to_csv(out_path, index=False)
    print(f"Saved {out_path} — {df.shape[0]} rows, {df.shape[1]} cols")

Saved /content/cleaned_data/customers_clean.csv — 99441 rows, 5 cols
Saved /content/cleaned_data/sellers_clean.csv — 3095 rows, 4 cols
Saved /content/cleaned_data/geolocation_clean.csv — 19011 rows, 5 cols
Saved /content/cleaned_data/category_translation_clean.csv — 71 rows, 2 cols
Saved /content/cleaned_data/products_clean.csv — 32951 rows, 10 cols
Saved /content/cleaned_data/orders_clean.csv — 99441 rows, 8 cols
Saved /content/cleaned_data/order_items_clean.csv — 112650 rows, 7 cols
Saved /content/cleaned_data/order_payments_clean.csv — 103886 rows, 5 cols
Saved /content/cleaned_data/order_reviews_clean.csv — 98410 rows, 7 cols


In [ ]:
import shutil
shutil.make_archive('/content/cleaned_data', 'zip', OUTPUT_DIR)

from google.colab import files
files.download('/content/cleaned_data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>